# 03 — QLoRA fine-tuning of Phi-3-mini on Colab Free (T4)

**What this notebook does**
1. Loads `microsoft/Phi-3-mini-4k-instruct` in 4-bit (NF4 + bf16 compute) via bitsandbytes.
2. Wraps it with a LoRA adapter (rank 16, alpha 32, q/k/v/o projections, dropout 0.05).
3. Loads the teacher labels at `/content/drive/MyDrive/trustlens/train.jsonl` and formats each row as a 3-turn chat (system + user + assistant-JSON).
4. Trains for 1 epoch with paged-AdamW-8bit, bf16, gradient checkpointing.
5. Saves the LoRA adapter back to `/content/drive/MyDrive/trustlens/phi3-trustlens-lora/`.
6. Sanity-checks the adapter on a handful of val rows.

**Before you run this notebook**
- Mount Drive and create the folder `MyDrive/trustlens/`.
- Upload `data/labeled/train.jsonl` and `data/processed/val.csv` from your Mac into that folder.
- Runtime → Change runtime type → **T4 GPU**.

Expected runtime: ~5–6 hours at 1000 examples × 1 epoch.

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1 datasets==3.0.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_DIR = '/content/drive/MyDrive/trustlens'
TRAIN_JSONL = f'{DRIVE_DIR}/train.jsonl'
VAL_CSV = f'{DRIVE_DIR}/val.csv'
ADAPTER_OUT = f'{DRIVE_DIR}/phi3-trustlens-lora'

assert os.path.exists(TRAIN_JSONL), f'upload teacher labels to {TRAIN_JSONL} first'
assert os.path.exists(VAL_CSV), f'upload val.csv to {VAL_CSV} first'
print('inputs ready.')

## Load Phi-3-mini in 4-bit

NF4 quantisation with double-quant gives the best memory / quality trade-off documented in the QLoRA paper. bf16 compute keeps gradients in a wider dtype than fp16 so we don't get NaNs on the T4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'microsoft/Phi-3-mini-4k-instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

mdl = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)

## Attach the LoRA adapter

We only train the q / k / v / o attention projections (rank 16). Everything else stays frozen at 4-bit. `prepare_model_for_kbit_training` re-enables input gradients and switches LayerNorms to fp32 — both required for stable QLoRA training.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

mdl = prepare_model_for_kbit_training(mdl)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

mdl = get_peft_model(mdl, lora_cfg)
mdl.print_trainable_parameters()

## Format the teacher labels as Phi-3 chat

Each row in `train.jsonl` becomes a 3-turn chat:
- **system**: the same CoT instruction the teacher saw.
- **user**: posting + numeric features + neighbor_fraud_rate (rebuilt the same way `build_user_prompt` does).
- **assistant**: the teacher's JSON answer.

We then apply Phi-3's chat template to get one big string and tokenize. `labels = input_ids` because this is causal LM SFT — the model learns to produce the assistant turn given system + user.

In [ ]:
import json

SYSTEM_PROMPT = '''You are a fraud-detection analyst reviewing online job postings.

You receive:
1. The full text of a job posting.
2. A few pre-computed numeric features (urgency words, all-caps ratio, etc.).
3. A neighbor_fraud_rate \u2014 the fraction of similar past postings that were fraud.

Think step-by-step before answering, in this order:
1. List suspicious signals you observe in the text.
2. Score each risk category (financial, legitimacy, data_privacy) from 0 to 100, higher = riskier.
3. Decide an action: "avoid", "caution", or "safe".
4. Output ONLY a single JSON object. No markdown fences, no preamble.

Output schema (exact keys):
{
  "trust_score": integer 0-100 (higher = safer),
  "red_flags": list of short strings,
  "risk_breakdown": {
    "financial": integer 0-100,
    "legitimacy": integer 0-100,
    "data_privacy": integer 0-100
  },
  "action": "avoid" or "caution" or "safe",
  "reasoning": 1-3 sentence explanation
}
'''

LABEL_KEYS = ['trust_score', 'red_flags', 'risk_breakdown', 'action', 'reasoning']

def build_user(job_text, feats, nfr):
    fb = '\n'.join(f'  {k}: {v:.2f}' for k, v in feats.items())
    return (f'Job posting:\n{job_text}\n\nPre-computed numeric features:\n{fb}\n'
            f'neighbor_fraud_rate: {nfr:.2f}\n\nOutput the JSON now.')

records = []
with open(TRAIN_JSONL) as f:
    for line in f:
        r = json.loads(line)
        feats = {k.replace('feat_', ''): v for k, v in r.items() if k.startswith('feat_')}
        records.append({'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user(r['job_text'], feats, r['neighbor_fraud_rate'])},
            {'role': 'assistant', 'content': json.dumps({k: r[k] for k in LABEL_KEYS})},
        ]})

print(f'{len(records)} training records')

In [ ]:
from datasets import Dataset

MAX_LEN = 2048

def to_features(ex):
    text = tok.apply_chat_template(ex['messages'], tokenize=False)
    out = tok(text, truncation=True, max_length=MAX_LEN, padding=False)
    out['labels'] = out['input_ids'].copy()
    return out

ds = Dataset.from_list(records).map(to_features, remove_columns=['messages'])
print(ds)

## Train

Per the locked spec:
- 1 epoch, lr 2e-4, batch 2, grad accum 8 (effective batch 16).
- bf16 compute, gradient checkpointing on (T4 has 15 GB; without checkpointing we OOM at this batch size).
- paged_adamw_8bit optimiser to keep optimiser state in 8-bit on the GPU.
- log every 20 steps, save every 200 steps to `/content/checkpoints/`.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    gradient_checkpointing=True,
    fp16=False,
    bf16=True,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = Trainer(
    model=mdl,
    args=args,
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)

trainer.train()

## Save adapter to Drive

In [ ]:
mdl.save_pretrained(ADAPTER_OUT)
tok.save_pretrained(ADAPTER_OUT)
print(f'adapter saved to {ADAPTER_OUT}')

## Smoke-test the adapter on 5 val rows

We pass the bare `description` column from val.csv (no precomputed features here — that needs ChromaDB, which lives on the Mac). The point of this cell is just to confirm the fine-tuned model **produces valid JSON in our schema**. Real metrics live in `src/evaluate.py` at STEP 14.

In [ ]:
import pandas as pd

val = pd.read_csv(VAL_CSV).head(5)
mdl.eval()

for i, row in enumerate(val.itertuples()):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user(row.job_text, {}, 0.0)},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt', truncation=True, max_length=3000).to(mdl.device)
    with torch.no_grad():
        gen = mdl.generate(**ids, max_new_tokens=512, do_sample=False)
    out = tok.decode(gen[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    print(f'--- val[{i}] (ground truth fraudulent={row.fraudulent}) ---')
    print(out[:600])
    print()

## Next steps

On your Mac, after the adapter directory is in Drive:
1. Download `MyDrive/trustlens/phi3-trustlens-lora/` to `models/phi3-trustlens-lora/` locally (or stay on Colab for inference).
2. `src/agent.py` (STEP 12) loads base + adapter and wraps as a LangChain `HuggingFacePipeline`.
3. `src/evaluate.py` (STEP 14) runs the full metric pass on the val set.